# Kaggriculture: V44 + Four Market Layers

## TL;DR

- **Live, V44 + layers 1, 2, 4** (submission 56277542): **2801 rating, rank about 188 of 9246 teams** on 16 Sep 2026 (a snapshot; ratings keep moving). It went **26-3** in its first 29 live games, and the three losses were by 1, 348 and 663 coins. Plain V44, submitted by us, plateaued around 2630-2657.
- **Live, all four layers** (submission 56280605): **22-2** in its first 24 live games, with mean final money of **115.9k vs 86.3k**. Its rating is still climbing (2389 after 4 hours), so it has not settled yet.
- **Local, exact engine dynamics, both seats per seed:** the four-layer build beats plain V44 **10-0** on two 5-seed sets (+4329 and +1607 mean margin) and pipe-5 **10-0** (+3737).
- **Live-opponent tapes:** we replayed the opponents' recorded actions from the three-layer build's 29 newest live games (3 open with BUY 70 / SELL 70 wheat on turn 1). An exact replay of the live three-layer file reproduces all 29 results to the coin (26-3). On the same tapes the four-layer build goes **27-2** (+6451 mean margin) and plain V44 goes 24-5 (+5415).
- **Caveat:** locally, the four-layer and three-layer builds are about equal (5-5 at +31, 7-3 at +14). The `USE_CADENCE` flag picks one. Local and tape results are filters, not guarantees of live results.
- **What this is:** the public V44 agent (V43 chassis lineage), unchanged, plus four small wrapper layers appended to the end of the file. Three change when and in what order stock is sold; one changes which animals to buy when a yarn store is open. The notebook builds the four-layer file by default; set `USE_CADENCE = False` for the three-layer file behind the 2801 rating.

## What this is

This notebook is a derivative of the public **V44** agent from this competition (V43 chassis lineage). The V44 file is included unchanged, and four small wrapper layers are added to the end of it. Each layer takes the previous entry point as its host (the last callable in the file), adjusts the host's action and falls back to the host's action if anything goes wrong. Everything that farms and plans comes from V44; the layers only adjust its orders. Three layers change when and in what order stock is sold, and one changes which animals to buy on seeds with a yarn store.

## The four layers at a glance

| # | Layer | What it does | When it acts |
|---|---|---|---|
| 1 | Preguard | Sells at hours 21-22 the stock that V44's own hour-23 storage guard would dump, so it sells one or two steps before a same-code rival | Hours 21-22, when the shed is about to overflow (6-unit margin) |
| 2 | Lockstep | Replays the engine's unit-by-unit market resolution to pick the best SELL order against an assumed exact copy of its own orders, and raises V44's race horizon for clone opponents from 8 to 9 | Only when V44's clone detector fires |
| 3 | Cadence | Holds back a few units just before a town consumption tick and sells them right after it | Steps just before a tick, from day 4 |
| 4 | Yarn herd | Buys sheep instead of cows and geese when YARN_STORE is unlocked | Animal purchases on days 8-11; otherwise it makes exactly V44's moves |

## Results

In all local tests each seed is played twice, once from each seat. "Margin" is our final money minus the opponent's, averaged per game.

**Live leaderboard**

| Build | Submission | Live record | Notes |
|---|---|---|---|
| V44 + layers 1, 2, 4 (three-layer) | 56277542 | 26-3 in first 29 games | 2801 rating, rank about 188 / 9246 (16 Sep 2026 snapshot); losses by 1, 348, 663 coins |
| V44 + layers 1, 2, 3, 4 (four-layer) | 56280605 | 22-2 in first 24 games | Mean final money 115.9k vs 86.3k; 2389 after 4 hours and still climbing |
| Plain V44 (our earlier submission) | - | - | Plateaued around 2630-2657 |

**Local, exact dynamics**

| Matchup | Seeds | W-L | Mean margin |
|---|---|---|---|
| Four-layer vs V44 | 47001-47005 | 10-0 | +4329 |
| Four-layer vs V44 | 46001-46005 | 10-0 | +1607 |
| Four-layer vs pipe-5 | 47001-47005 | 10-0 | +3737 |
| Three-layer vs V44 | 46001-46010 | 20-0 | +1902 |
| Three-layer vs pipe-5 | 5 seeds | 10-0 | +2105 |
| Three-layer vs pipe-4 | 5 seeds | 10-0 | +2060 |
| Three-layer vs Farming Score V5 | 5 seeds | 10-0 | +2646 |
| Four-layer vs three-layer | 5 seeds | 5-5 | +31 |
| Four-layer vs three-layer | 5 seeds | 7-3 | +14 |

Plain V44 also beats pipe-4 and pipe-5 locally, so the margins against them are build-vs-agent margins, not the layers' own gain.

Official `kaggle_environments` runner, four-layer vs V44, seed 46001: **95634 vs 93549** in each seat. The slowest single agent call took 197 ms, and the mean was 1.4 ms.

**Live-opponent tape testbed** (opponents' recorded actions from our live replays, replayed exactly with the same seed and seat)

| Tape set | Plain V44 | Three-layer | Four-layer |
|---|---|---|---|
| 62 tapes, older live games | 61-1 (+9953) | 61-1 (+10827) | 61-1 (+10694) |
| 29 tapes, newest live games (incl. BUY 70 / SELL 70 openers) | 24-5 (+5415) | 26-3 (+6399) | 27-2 (+6451) |

## About the base

- **V44 is a public agent from this competition**, built on the V43 chassis lineage. The planning, farming and trading all come from V44, including the race logic, clone detection and day-end storage guard that our layers hook into.
- **pipe-4 and pipe-5** are related public agents from the same family. We used them as test opponents, along with Farming Score V5.
- **This notebook is a derivative work.** The V44 source is embedded byte-for-byte (sha256 prefix `797d9bca`, original line endings kept), including its Apache-2.0 license text and attribution notices. Our contribution is the four added layers and the testing described below.

## How to use

1. Set `USE_CADENCE` at the top of the **Build the submission** cell (`True` = four-layer, the default; `False` = three-layer).
2. **Save Version -> Save & Run All.** The notebook writes `/kaggle/working/main.py` and `/kaggle/working/submission.tar.gz`.
3. Submit `submission.tar.gz` (or `main.py`) from the notebook's Output, or download it and upload it on the competition's Submit page. The Output also lists the `build/*.py` parts; those are not submissions.
4. The build cell asserts the full sha256 of `main.py`, so a clean run means you have exactly the file we run live. If you edit a layer, the assert fails by design: set `VERIFY_LIVE_SHA = False` (or update `EXPECTED_SHA256`).

The agent is the **last callable** in `main.py`. If you add code, put it at the end and keep your agent function last.

## How we evaluate

Given a seed and both players' actions, a Kaggriculture game always plays out the same way. We build two test tools on that fact, and we lean on mirror games as the main filter.

### 1. Exact-dynamics fast simulator

On every step, the official `kaggle_environments` runner deep-copies the state, validates observations and actions against the schema, and records history. For testing we call the `kaggriculture` interpreter directly and skip that overhead:

- The interpreter is the same, so market, town and end-of-day code, and therefore the game dynamics, are exact.
- It gives **identical final money** to the official runner.
- It is **much faster**, which makes it cheap to test each idea on many seeds.

Every candidate goes through the same protocol:

- Each seed is played twice, once from each seat.
- Results are confirmed on fresh seed ranges that were not used for tuning.
- Opponents are plain V44 (the mirror), near-copies with different settings (pipe-4, pipe-5) and other public agents (Farming Score V5).
- Before shipping, we package `main.py` and load it with Kaggle's file loader, which confirms the last callable is the wrapper. Then we run it in the official runner and record per-call timing.

### 2. Live-opponent tape testbed

Local tests are limited to the public agents you have, and the ladder has opponents you do not have. So we bring their moves home:

1. Download the replays of our own live games.
2. From each replay, extract the opponent's action at every step (its "tape").
3. Wrap the tape in a replay agent that returns the recorded action for the current step, unchanged, with no end-of-game overrides.
4. Play the candidate against the tape with the same seed and seat as the live game.

**Sanity check:** replaying the live three-layer file against the tapes of its own 29 newest games reproduces both players' final money to the coin in all 29 (26-3, the live record). An earlier replay agent that sold the opponent's whole shed at step 718 cost the opponent up to about 3000 coins per game (median 382) and turned all three live losses into tape wins, so exact replay matters.

For each candidate we report the win-loss record, the mean margin, **losses converted** (tapes the baseline lost and the candidate wins) and **wins kept** (tapes the baseline won and the candidate still wins). A change that converts losses but gives up wins is rejected.

**Limitation:** a tape does not react to us. If the candidate moves the market, the opponent still sends its recorded orders, and they now fill at different prices or fail. Tapes are a strong filter against real opponent behavior, but they do not replace live games.

### 3. Why mirror games against V44 are decisive

V44 against itself makes the same decisions in the same situations, and on most seeds the game ends in an **exact tie**. That makes the mirror a zero-noise test:

- Any change shows up at once as a non-zero margin, and the sign means something even when the gain is small. Layer 3 gains only +30 to +70 per game, yet it went 64-2 over 66 mirror games.
- Regressions cannot hide behind variance.
- It is a frequent matchup on the ladder (see the next section).

The risk is overfitting to an exact copy. Every mirror gain is re-checked on fresh seeds, against pipe-4 and pipe-5, and on live tapes.

### 4. Why clone-vs-clone games matter

V43 and then V44 were public, strong, single-file agents, and pipe-4 and pipe-5 come from the same family. When plain V44 plays the exact tapes of our live opponents, its own clone detector fires in 24 of the 29 newest games and 34 of the 62 older ones, so clone-vs-clone games are common on the ladder.

When two clones meet, their farming is nearly identical, so the market decides the game. The engine resolves both players' market orders **slot by slot and unit by unit**. For order slot *i*, both players get a price for their next unit from the same market inventory, both sales go through, and the price moves after every unit. This has three consequences:

- When two clones sell the same product in the same slot on the same turn, they share the price drop evenly.
- A player who sells a contested product one step earlier, or in an earlier slot than the rival, gets the undepressed price and leaves the lower price to the rival.
- Town shops remove units every 4 steps and the town center every 24, so timing relative to those ticks matters.

Layers 1-3 target exactly these same-turn collisions. Layer 4 changes a herd choice that clones do not change. This also helps explain why plain V44 plateaued: against copies of itself, it has no edge.

## V44 base

The base is the public V44 agent from this competition (V43 chassis lineage), used unchanged. Its exact `main.py` (327,314 bytes) is embedded in the next cell as a gzip + base64 blob. The cell decodes it, asserts the full sha256

`797d9bca309d481e18cdbb675ce6d6de9967d266cbafb4c2352c68fb10264e2f`

and writes the bytes to `build/v44_base.py`. The file keeps its own Apache-2.0 license text and attribution notices; our changes are only the layers appended after it.

The cell also switches to `/kaggle/working` (or to the current directory outside Kaggle), so the `%%writefile` cells below write into `build/`.

In [ ]:
import base64, gzip, hashlib, os

# Work in /kaggle/working on Kaggle, or in the current directory anywhere else.
WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd()
os.chdir(WORK)
os.makedirs('build', exist_ok=True)

V44_SHA256 = '797d9bca309d481e18cdbb675ce6d6de9967d266cbafb4c2352c68fb10264e2f'

# The public V44 main.py, gzip-compressed and base64-encoded.
V44_GZ_B64 = (
    "..."
)

v44_bytes = gzip.decompress(base64.b64decode(V44_GZ_B64))
assert hashlib.sha256(v44_bytes).hexdigest() == V44_SHA256, 'V44 blob does not match its sha256'
with open(os.path.join('build', 'v44_base.py'), 'wb') as f:
    f.write(v44_bytes)
print(f'V44 base: {len(v44_bytes):,} bytes, sha256 {V44_SHA256} OK')
print('written to', os.path.join(WORK, 'build', 'v44_base.py'))

## Layer 1 - Preguard sells at hours 21-22

**What V44 does.** When hour 23 ends, the engine drops every unit's carried items into the shed and destroys anything above the 100-item cap. V44 guards against this with a day-end storage guard (EXP-154). At hour 23, if shed plus carried items exceed 99, it sells the excess, highest price first. V43/V44-lineage rivals (pipe-4, pipe-5) run the same guard, so they often dump the same lots on the same step.

**What the layer changes.** On hours 21 and 22 of days 1-28, it projects that overflow using V44's own helpers: `_r127_fields` applies this step's unit actions and `_r97_market_stock` applies the orders already queued. Day 29 is left to V44's terminal planner. In the guard's price order, the layer then adds SELLs for MILK, STRAWBERRY, MELON, WOOL and TOMATO, skipping any priced below 2. The hour-23 guard stays in place as a backstop.

**Why it gains.** Within a step, the market runs before town consumption.

| Hour | Consumption after the market | Seller |
|---|---|---|
| 20 | shops | |
| 21, 22 | none | this layer |
| 23 | none, then end-of-day drop | V44 guard |
| 0 | shops + town center | |

No consumption runs between the hour-20 and hour-0 markets, so a passive market pays the same price at hours 21, 22 and 23. The market is lockstep: both players quote each unit at the same inventory, and each sale adds to supply and pushes the price down. Whoever sells first gets the top of the price curve, and a rival's hour-23 dump starts lower.

**Margin 6.** The projection can't see later harvests or deposits, so the layer sells 6 units past it (`_Y_MARGIN=-6`: target 93, not 99). That usually leaves the guard nothing to dump. Six was the best buffer we tried.

**Evidence (layer alone on V44, fresh seeds).** 38-2 against plain V44 (both losses by 6 coins), mean margins +562 to +834. Against pipe-5: +1400 to +2423 mean margin (plain V44 also beats pipe-5 on those seeds, so only part of that is the layer).

**Limitations.** This is a race edge. The gain is small against opponents that don't dump at hour 23, and the buffer can sell a few units early. Products outside the list count toward the overflow but are never pre-sold, so the guard sells them.

This block is appended right after V44's source. It takes the previous last callable as its host and adds nothing but SELL orders to that host's market list.

In [ ]:
%%writefile build/layer1_preguard.py

# ---- v44y pre-guard: quote at hour 21,22 what the hour-23 day-end storage guard (EXP-154) would dump ----
# The guard sells shed stock by price desc once shed+carried exceeds 99 at hour 23. Selling those lots
# a step earlier stays in the same town-consumption price window and quotes before a same-tape rival.
_PG_HOST=[v for v in list(globals().values()) if callable(v)][-1]
_Y_HOURS=(21, 22)
_Y_ITEMS=('MILK', 'STRAWBERRY', 'MELON', 'WOOL', 'TOMATO')
_Y_MIN_DAY=1
_Y_MARGIN=-6
_PG_REPORT={'preguard_turns':0,'preguard_units':0,'preguard_errors':0}

def _y_preguard(obs,action):
    step=int(obs['step'])
    if step%24 not in _Y_HOURS or step//24<_Y_MIN_DAY or step>=696:return action
    orders=[list(o) for o in (action.get('market') or [])]
    if len(orders)>=10:return action
    farm,private=_r127_fields(obs,action)
    stock,_,_=_r97_market_stock(private['shed'],orders)
    carried=sum(max(0,int(n)) for bag in private['inventories'] for n in bag.values())
    needed=sum(max(0,int(v)) for v in stock.values())+carried-99-_Y_MARGIN
    if needed<=0:return action
    prices=obs['market']['prices'];extra=[]
    for item in sorted(PRODUCTS,key=lambda it:-int(prices.get(it,0))):
        avail=max(0,int(stock.get(item,0)));qty=min(needed,avail)
        if qty<=0:continue
        if item in _Y_ITEMS and int(prices.get(item,0))>=2:extra.append(['SELL',item,qty])
        needed-=qty
        if needed<=0:break
    if not extra or len(orders)+len(extra)>10:return action
    _PG_REPORT['preguard_turns']+=1;_PG_REPORT['preguard_units']+=sum(o[2] for o in extra)
    return dict(action,market=orders+extra)

def agent_v44y_preguard(observation,configuration=None):
    action=_PG_HOST(observation,configuration)
    try:
        standard=configuration is None or all(configuration.get(k,v)==v for k,v in [('boardSize',10),('turnsPerDay',24),('shedCapacity',100),('maxMarketOrdersPerTurn',10)])
        if standard:action=_y_preguard(observation,action)
    except Exception:_PG_REPORT['preguard_errors']+=1
    return action
agent_v44y_preguard.telemetry=_PG_REPORT



## Layer 2 - Lockstep SELL ordering and clone horizon 9

**How the engine clears the market.** After unit actions, `_process_market` walks both order lists slot by slot. Inside a slot, SELL and BUY_PRODUCT run in lockstep, one unit at a time: both players are quoted at the same pre-commit inventory, then both commit, and each SELL unit priced above $1 adds one unit of inventory, lowering the next quote. Within a slot nobody is first.

**Why clones collide.** Two V44 copies (or a rival running the same public route tape) emit identical market lists, so they sell the same product in the same slot and split every dump unit by unit. The only lever is slot order: if my glutted product sits where the clone still sells something else, I take the pre-dump prices and its copy lands on the inventory I raised. V44's R37 sort only reorders SELL runs of distinct products.

**What the layer changes.** From step 216, when V44's own race detector flags a clone this turn (farmer and hand positions equal to the rival's on at least 4 of the last 6 turns, plus at least 95% matching occupied tiles; from step 696, the stored history plus a fresh tile check), each contiguous run of 2-6 SELL orders is permuted. Every distinct permutation is replayed through a copy of the engine lockstep against the clone's assumed list (the host's unmodified list and the same projected shed, i.e. an exact copy of us), keeping the order that maximizes own revenue minus clone revenue. The original order scores exactly 0, so only a positive edge changes anything. Quantities never change.

**Horizon 8 -> 9.** In clone mode V44 pre-sells shed stock its tape plans to sell within 8 turns, and it escalates to 24 after it detects a lost race; we keep that. At 9 we sell one turn before a horizon-8 clone.

| Variant vs plain V44 | Result |
|---|---|
| reorder + horizon 9 | 20-0, +1436 (seeds 46001-46010) |
| reorder only | 19-1, +529 |
| horizon 12 only | lost a seed by 2719 |
| horizon 24 | too aggressive |

**Limits.** It assumes an exact clone, so a gated rival with different lists may not yield the edge. The replay ignores cash and shed caps on BUY_PRODUCT. A 6-order block means 720 replays; the full agent peaked at 197 ms.

Appended after Layer 1: `_V44Y_HOST` captures the previous last callable (`agent_v44y_preguard`), the module-level `_RACE_HORIZON_CLONE = 9` overrides V44's 8, and `v44y_lockstep_agent` becomes the new entry point.

In [ ]:
%%writefile build/layer2_lockstep.py

# ---- v44y: clone-mode race horizon override ----
_RACE_HORIZON_CLONE = 9

# ---- v44y: exact lockstep best-response SELL ordering against a detected clone ----
_V44Y_HOST = [v for v in list(globals().values()) if callable(v)][-1]
_V44Y_REORDER_GATE = True
_V44Y_REPORT = dict(v44y_reorder_turns=0, v44y_reorder_gain=0.0, v44y_errors=0)
import itertools as _v44y_it

def _v44y_price(item, inventory, params):
    return _r37_market_price(item, inventory, params)

def _v44y_params(obs):
    params = {k: dict(v) for k, v in _R37_MARKET_PARAMS.items()}
    for k, patch in (obs['market'].get('params') or {}).items():
        if k in params and isinstance(patch, dict): params[k].update(patch)
    return params

def _v44y_lockstep(orders_me, orders_opp, inv0, stock_me, stock_opp, params):
    """Replay the engine's per-slot / per-unit lockstep for SELL and BUY_PRODUCT orders (money-unbounded).
    Returns (revenue_me, revenue_opp)."""
    inv = dict(inv0); stock = [dict(stock_me), dict(stock_opp)]; rev = [0.0, 0.0]
    queues = [list(orders_me), list(orders_opp)]
    for i in range(max(len(queues[0]), len(queues[1]))):
        rem = [None, None]
        for p in (0, 1):
            if i < len(queues[p]):
                o = queues[p][i]
                if o and len(o) >= 3 and o[0] in ('SELL', 'BUY_PRODUCT') and o[1] in params:
                    try: n = int(o[2])
                    except Exception: n = 0
                    if n > 0: rem[p] = [o[0], o[1], n]
        guard = 0
        while True:
            guard += 1
            if guard > 5000: break
            quoted = [None, None]
            for p in (0, 1):
                r = rem[p]
                if r is None or r[2] <= 0: continue
                if r[0] == 'SELL':
                    quoted[p] = ('SELL', r[1], _v44y_price(r[1], inv[r[1]], params))
                elif r[1] in ('WHEAT', 'FERTILIZER'):
                    quoted[p] = ('BUY_PRODUCT', r[1], _v44y_price(r[1], inv[r[1]] - 1, params))
                else:
                    rem[p] = None
            if quoted[0] is None and quoted[1] is None: break
            committed = False
            for p in (0, 1):
                q = quoted[p]
                if q is None: continue
                op, item, price = q
                if op == 'SELL':
                    if stock[p].get(item, 0) <= 0:
                        rem[p] = None; continue
                    stock[p][item] -= 1; rev[p] += price
                    if price > 1: inv[item] += 1
                else:
                    stock[p][item] = stock[p].get(item, 0) + 1; rev[p] -= price; inv[item] -= 1
                rem[p][2] -= 1; committed = True
            if not committed: break
    return rev[0], rev[1]

def _v44y_reorder(obs, action):
    market = action.get('market') or []
    if len(market) < 2: return action
    orders = [list(o) if isinstance(o, (list, tuple)) else o for o in market]
    blocks = []; i = 0
    while i < len(orders):
        o = orders[i]
        if o and o[0] == 'SELL':
            j = i
            while j < len(orders) and orders[j] and orders[j][0] == 'SELL': j += 1
            if 2 <= j - i <= 6: blocks.append((i, j))
            i = j
        else: i += 1
    if not blocks: return action
    view = FarmView(obs)
    stock = projected_shed(action, view)
    stock = {k: max(0, int(v)) for k, v in stock.items()}
    params = _v44y_params(obs)
    inv0 = {k: int(v) for k, v in obs['market']['inventory'].items()}
    opp = [list(o) for o in orders]
    def margin(cand):
        a, b = _v44y_lockstep(cand, opp, inv0, stock, stock, params)
        return a - b
    base = margin(orders); best = base; best_orders = None
    for (i, j) in blocks:
        blk = orders[i:j]; n = j - i
        seen = set()
        for perm in _v44y_it.permutations(range(n)):
            key = tuple((blk[p][1], int(blk[p][2])) for p in perm)
            if key in seen: continue
            seen.add(key)
            cand = orders[:i] + [blk[p] for p in perm] + orders[j:]
            v = margin(cand)
            if v > best + 0.5: best = v; best_orders = cand
        if best_orders is not None:
            orders = best_orders; best_orders = None
    if best <= base + 0.5: return action
    _V44Y_REPORT['v44y_reorder_turns'] += 1; _V44Y_REPORT['v44y_reorder_gain'] += best - base
    out = dict(action); out['market'] = orders
    return out

def _v44y_clone_gate(obs):
    player = int(obs['player']); step = int(obs['step'])
    st = _RACE_STATE.get(player) or {}
    if st.get('horizon', 0) > 0: return True
    if step >= 696 and len(st.get('hist', [])) >= 4 and sum(st['hist']) >= 4:
        return _r37_similarity(obs) >= .95
    return False

def v44y_lockstep_agent(observation, configuration=None):
    action = _V44Y_HOST(observation, configuration)
    try:
        step = int(observation.get('step', 0))
        if step >= 216 and (not _V44Y_REORDER_GATE or _v44y_clone_gate(observation)):
            action = _v44y_reorder(observation, action)
    except Exception:
        _V44Y_REPORT['v44y_errors'] += 1
    return action
v44y_lockstep_agent.telemetry = _V44Y_REPORT


## Layer 3 - Cadence (sell timing around town ticks)

**Engine timing.** Each step the market clears first, then the town consumes: when `step % 4 == 0` every unlocked shop removes 1 unit per product (2 for single-product shops), plus 1 of everything every 24 steps. Orders are paired by list index across both players and filled one unit at a time at a shared quote.

**What V44 does.** V44 (V43 chassis) replays a route tape and wins the same-turn sale race by selling planned lots as soon as the stock exists.

**What this layer changes.** At phases `step % 4` in {0, 2, 3} (steps 96-716, not hour 23) it trims `floor(n/2)` units from a host SELL of CARROT, TOMATO, STRAWBERRY, MELON, MILK or WOOL, where `n` is that item's relief at the next tick. Held units are appended as a separate SELL after the host's orders at the first post-tick step (phase 1) where the host is not selling that item; after 3 steps, on shed pressure, or from step 717 they go out regardless. No deferral when the route tape sells that item within 8 steps, when shed room is short (DROP overflow is destroyed), or when this step's purchases plus 300 would not stay funded (money must also be at least 1000).

**Why it gains and naive versions lose.** Against a clone, with price drop `s` per unit sold:

| Change | Margin vs clone |
|---|---|
| Hold `d` units across a tick, sell alone | `+s·d(n-d)`, best near `n/2` |
| Split a lot, no tick between | `-s·d²` |
| Delay a whole lot `q > n` | `s·q(n-q) < 0` |
| Release where the clone sells `q_s` too | extra `-2s·q_s·d` |

`n` is a few units, so the edge is small but nearly always positive.

**Evidence.** Layer 3 alone vs plain V44: 64-2 over 66 mirror games, +30 to +70 per game; splits and whole-lot delays lost.

**Limitations.** A tie-breaker for clone-heavy lobbies, not a money engine. On top of layers 1, 2, 4 it is practically neutral: four-layer vs three-layer 5-5 (+31) and 7-3 (+14); on the older 62 tapes both go 61-1 (+10694 vs +10827), and on the newest 29 it goes 27-2 vs 26-3. Errors or a non-standard config return the host action unchanged.

Below is the exact Layer 3 block, appended after Layer 2. It captures the previous entry as `_CD_HOST`, and its last function, `_v44y_sell_cadence_agent`, becomes the new agent.

In [ ]:
%%writefile build/layer3_cadence.py


# ---------------------------------------------------------------- v44y sell-cadence wrapper (v3: hold & release)
_CD_HOST = [v for v in list(globals().values()) if callable(v)][-1]
_V44Y_CFG = {'rule': 'floor', 'items': ['CARROT', 'TOMATO', 'STRAWBERRY', 'MELON', 'MILK', 'WOOL'], 'window': [96, 717], 'phases': [0, 2, 3], 'min_lot': 1, 'cash_margin': 300, 'money_floor': 1000, 'shed_cap': 100, 'max_hold': 3, 'hard_hold': 3, 'final_step': 717, 'lookahead': 8}
_V44Y_STATE = {}
_V44Y_SHOPS = {"BAKERY": ["EGG", "WHEAT"], "PIZZA_SHOP": ["MILK", "TOMATO", "WHEAT"], "BRUNCH_SPOT": ["EGG", "WHEAT", "STRAWBERRY"], "YARN_STORE": ["WOOL"], "ICE_CREAM_SHOP": ["STRAWBERRY", "MILK", "WHEAT"], "PET_CAFE": ["CARROT"], "SMOOTHIE_SHOP": ["STRAWBERRY", "MILK"], "FARMERS_MARKET": ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY"]}
_V44Y_SEED_PRICE = {"WHEAT": 10, "CARROT": 20, "TOMATO": 50, "STRAWBERRY": 100, "MELON": 80}
_V44Y_ANIMAL_COST = {"GOOSE": 300, "COW": 400, "SHEEP": 500}
_V44Y_LAND = (1000, 2000, 4000)
_CD_REPORT = {"deferred_units": 0, "deferred_lots": 0, "released_alone": 0, "released_merged": 0, "released_forced": 0,
                "absorbed_by_host": 0, "blocked_shed": 0, "blocked_cash": 0, "errors": 0}


def _v44y_fib(n):
    a, b = 1, 1
    for _ in range(max(0, n)):
        a, b = b, a + b
    return a


def _v44y_relief_at(tick_step, shops):
    """Units the town removes per product right after the market of `tick_step` (a multiple of 4)."""
    out = {}
    for s in shops:
        prods = _V44Y_SHOPS.get(s, [])
        m = 2 if len(prods) == 1 else 1
        for p in prods:
            out[p] = out.get(p, 0) + m
    if tick_step % 24 == 0:
        for p in ("WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON", "EGG", "MILK", "WOOL"):
            out[p] = out.get(p, 0) + 1
    return out


def _v44y_buy_cost(market, prices, hires_today, quadrants):
    cost = 0.0
    hires = 0
    q = quadrants
    for o in market:
        if not o:
            continue
        op = o[0]
        if op == "HIRE":
            cost += _v44y_fib(hires_today + hires)
            hires += 1
        elif op == "BUY_LAND":
            i = q - 1
            if 0 <= i < 3:
                cost += _V44Y_LAND[i]
                q += 1
        elif len(o) >= 3:
            n = max(0, int(o[2]))
            if op == "BUY_SEED":
                cost += _V44Y_SEED_PRICE.get(o[1], 0) * n
            elif op == "BUY_ANIMAL":
                cost += _V44Y_ANIMAL_COST.get(o[1], 0) * n
            elif op == "BUY_PRODUCT":
                cost += prices.get(o[1], 0) * n
    return cost


def _v44y_append_sell(market, item, qty):
    """Append a separate SELL after the host's orders (keeps the host's order indices paired with a clone's)."""
    if len(market) >= 10:
        for o in market:
            if o and o[0] == "SELL" and len(o) >= 3 and o[1] == item:
                o[2] = int(o[2]) + qty
                return "merged"
        return None
    market.append(["SELL", item, qty])
    return "alone"


def _v44y_merge_sell(market, item, qty):
    for o in market:
        if o and o[0] == "SELL" and len(o) >= 3 and o[1] == item:
            o[2] = int(o[2]) + qty
            return "merged"
    return _v44y_append_sell(market, item, qty)


def _v44y_defer_amount(rule, lot, n):
    if n <= 0:
        return 0
    if rule == "half":
        return min(lot, (n + 1) // 2)
    if rule == "n":
        return min(lot, n)
    if rule == "floor":
        return min(lot, n // 2)
    if rule == "all":
        return lot
    return 0


def _v44y_stock_view(observation, action, shed):
    """(projected shed after this step's unit actions, units the shed may still receive next step)."""
    try:
        view = FarmView(observation)
        proj = projected_shed(action, view)
        carried = sum(max(0, int(n or 0)) for inv in view.invs for n in inv.values())
        produced = 0
        units = [action.get("farmer") or ["PASS"]] + list(action.get("hands") or [])
        for i in range(min(len(units), len(view.positions))):
            a = units[i]
            if not a:
                continue
            tile = _tile_at(view.tiles, view.positions[i])
            if a[0] == "HARVEST" and isinstance(tile, dict):
                produced += max(0, int(tile.get("yield_units", 0) or 0))
            elif a[0] == "COLLECT_FERTILIZER" and isinstance(tile, dict) and tile.get("fertilizer_available"):
                produced += 1
        return {k: max(0, int(v)) for k, v in proj.items()}, carried + produced
    except Exception:
        _CD_REPORT["errors"] += 1
        return dict(shed), 0


def _v44y_sell_cadence_agent(observation, configuration=None):
    action = _CD_HOST(observation, configuration)
    try:
        cfg = _V44Y_CFG
        step = int(observation.get("step", 0) or 0)
        player = int(observation.get("player", 0) or 0)
        st = _V44Y_STATE.get(player)
        if st is None or step <= st["step"]:
            st = _V44Y_STATE[player] = {"step": -1, "held": {}, "since": {}}
        st["step"] = step
        if configuration is not None and not all(configuration.get(k, v) == v for k, v in (("boardSize", 10), ("turnsPerDay", 24), ("shedCapacity", 100), ("maxMarketOrdersPerTurn", 10))):
            return action
        if not isinstance(action, dict):
            return action
        market = [list(o) for o in (action.get("market") or [])]
        farms = observation.get("farms") or []
        me = farms[player] if player < len(farms) else {}
        shed = {k: int(v or 0) for k, v in dict((observation.get("private") or {}).get("shed") or {}).items()}
        prices = dict((observation.get("market") or {}).get("prices") or {})
        shops = list((observation.get("town") or {}).get("unlocked_shops") or [])
        money = float(me.get("money", 0) or 0)
        proj, incoming = _v44y_stock_view(observation, action, shed)
        host_sell = {}
        for o in market:
            if o and o[0] == "SELL" and len(o) >= 3:
                host_sell[o[1]] = host_sell.get(o[1], 0) + max(0, int(o[2]))
        # 1. reconcile the ledger with what the host will leave in the shed after its own orders
        held = st["held"]
        for it in list(held.keys()):
            left = max(0, proj.get(it, 0) - host_sell.get(it, 0))
            if held[it] > left:
                _CD_REPORT["absorbed_by_host"] += held[it] - left
                held[it] = left
            if held[it] <= 0:
                held.pop(it, None)
                st["since"].pop(it, None)
        # stock after the host's orders (bounded by stock) and the room left for next step's drops
        avail = dict(proj)
        revenue = 0.0
        for o in market:
            if o and o[0] == "SELL" and len(o) >= 3:
                x = min(max(0, int(o[2])), avail.get(o[1], 0))
                avail[o[1]] = avail.get(o[1], 0) - x
                revenue += prices.get(o[1], 0) * x
        total_after = sum(avail.values())
        room = cfg["shed_cap"] - total_after - incoming
        # 2. release held lots
        for it in list(held.keys()):
            q = held[it]
            age = step - st["since"].get(it, step)
            forced = room < 0 or step >= cfg["final_step"]
            if host_sell.get(it, 0) <= 0:
                if step % 4 == 1 or age >= cfg["max_hold"] or forced:
                    how = _v44y_append_sell(market, it, q)
                    if how:
                        _CD_REPORT["released_alone" if how == "alone" else "released_merged"] += q
                        held.pop(it, None); st["since"].pop(it, None); room += q
            else:
                if age >= cfg["hard_hold"] or forced:
                    how = _v44y_merge_sell(market, it, q)
                    if how:
                        _CD_REPORT["released_forced"] += q
                        held.pop(it, None); st["since"].pop(it, None); room += q
        # 3. new deferrals from this step's lots
        lo, hi = cfg["window"]
        if lo <= step < hi and step % 24 != 23 and (step % 4) in cfg["phases"] and money >= cfg["money_floor"]:
            next_tick = step if step % 4 == 0 else step + (4 - step % 4)
            relief = _v44y_relief_at(next_tick, shops)
            planned_soon = set()
            la = int(cfg.get("lookahead", 0) or 0)
            if la > 0:
                try:
                    route = _IMPL.chassis.players[player]["route"]
                    tape = _IMPL.chassis.routes[route]
                    for t in range(step + 1, min(step + 1 + la, len(tape))):
                        for o in (tape[t].get("market") or []) if isinstance(tape[t], dict) else []:
                            if o and o[0] == "SELL" and len(o) >= 3 and int(o[2]) > 0:
                                planned_soon.add(o[1])
                except Exception:
                    _CD_REPORT["errors"] += 1
            cost = _v44y_buy_cost(market, prices, int(me.get("hires_today", 0) or 0), len(me.get("unlocked_quadrants") or []))
            slack = money + revenue - cost - cfg["cash_margin"]
            for o in market:
                if not (o and o[0] == "SELL" and len(o) >= 3):
                    continue
                it = o[1]
                q = int(o[2])
                if it not in cfg["items"] or q <= 0 or prices.get(it, 0) < 2 or it in planned_soon:
                    continue
                lot = min(q, proj.get(it, 0))
                if lot < cfg["min_lot"]:
                    continue
                d = min(lot, _v44y_defer_amount(cfg["rule"], lot, relief.get(it, 0)))
                if d <= 0:
                    continue
                if d > room:
                    _CD_REPORT["blocked_shed"] += 1
                    continue
                if prices.get(it, 0) * d > slack:
                    _CD_REPORT["blocked_cash"] += 1
                    continue
                slack -= prices.get(it, 0) * d
                room -= d
                o[2] = q - d
                held[it] = held.get(it, 0) + d
                st["since"].setdefault(it, step)
                _CD_REPORT["deferred_units"] += d
                _CD_REPORT["deferred_lots"] += 1
        action["market"] = market[:10]
        return action
    except Exception:
        _CD_REPORT["errors"] += 1
        return action


_v44y_sell_cadence_agent.telemetry = _CD_REPORT


## Layer 4 - Yarn herd (shop-aware herd)

**Shop demand model.** A town shop is drawn on days 3, 6, ... 24 (8 draws, with replacement). Every 4 steps each shop instance removes one unit of each product it lists from the shared market; single-product shops (`YARN_STORE`, `PET_CAFE`) remove two. One yarn store drains 12 wool per day, a milk shop only 6 milk. Otherwise only the town center buys wool, 1 per day. Wool rises only slowly on scarcity (log shape: about $240, +20% over its $200 base, after a 105-unit shortage) but collapses on glut (`sq` shape: about 59 surplus units reach the $1 floor). A sheep costs more than a cow (500 vs 400) and yields less often (every 3 days vs 2), so sheep usually lose to cows unless a yarn store absorbs the wool.

**What V44 does.** At step 144 it picks a route from the first two shops: a yarn store there selects the sheep-heavy V39 tape, otherwise an EXP240 tape buys geese (and on some routes a cow) on days 8-11. V44 has the mirror rule (day-9 sheep to cow with 2+ milk shops and no yarn store), but nothing for a yarn store drawn third on day 9.

**What the layer changes** (days 8-11, `YARN_STORE` unlocked):
1. `BUY_ANIMAL COW/GOOSE` (qty 1-2) becomes `SHEEP` if estimated cash after all orders stays at 100 or more.
2. Sheep confirmed in the shed are credited; the tape's `PICKUP`/`PLACE` are rewritten to sheep, `BUILD_COOP` to `BUILD_PASTURE`.
3. Wool harvested at swapped sites is added to the tape's existing `SELL WOOL` orders. No new orders are created, so the order slots stay aligned with a V44 clone's list.

**Evidence.** On yarn-store seeds: 18-0 (+3770 mean) vs V44. When no swap happens, its output was action-identical to its host in our tests. Layer 4 is in both live builds, so the three-layer vs four-layer comparison says nothing about it.

**Limitations.** Fires only when a yarn store is open at the day 8-11 purchases (in practice when it is the third shop; a yarn store among the first two already routes V44 to its sheep-heavy tape). Extra wool also lowers the shared wool price, so with one yarn store margins can be thin, and an opponent selling no wool avoids that cost. After a goose swap, later coops are built as pastures (harmless for V44, which buys all its animals by day 11).

Layer 4 is appended last (after Layer 3, or after Layer 2 in the three-layer build). It takes the previous entry as `_Y_HOST` and passes the host's action through unchanged unless a yarn store is open at the day 8-11 animal purchases. On any exception it returns the host's action.

In [ ]:
%%writefile build/layer4_yarnherd.py

# ==== v44y shop-aware herd wrapper (appended after the public V44 file; host entry captured first) ====
_Y_HOST=[v for v in list(globals().values()) if callable(v)][-1]
_Y_CFG={'yarnsheep': True, 'yarngeese': True, 'days': (8, 11)}
import copy as _y_copy
_Y_PRODUCT={'COW':'MILK','SHEEP':'WOOL','GOOSE':'EGG'}
_Y_STRUCT={'COW':'PASTURE','SHEEP':'PASTURE','GOOSE':'COOP'}
_Y_COST={'COW':400,'SHEEP':500,'GOOSE':300}
_Y_SEED={'WHEAT':10,'CARROT':20,'TOMATO':50,'STRAWBERRY':100,'MELON':80}
_Y_STATES={}
_Y_REPORT={'swaps':0,'pick':0,'place':0,'harvest':0,'boost':0,'errors':0,'coop':0,'declined':0}

def _y_new_state():
    return {'last':-1,'pending':[],'credit':{},'sites':{},'sale':{},'coop_swap':0}

def _y_target(kind,shops,cfg):
    yarn='YARN_STORE' in shops
    egg=('BAKERY' in shops) or ('BRUNCH_SPOT' in shops)
    milk=sum(s in ('PIZZA_SHOP','ICE_CREAM_SHOP','SMOOTHIE_SHOP') for s in shops)
    if kind=='SHEEP' and cfg.get('nosheep') and not yarn and milk>=cfg.get('min_milk',0):return 'COW'
    if kind=='COW' and cfg.get('yarnsheep') and yarn:return 'SHEEP'
    if kind=='GOOSE' and cfg.get('nogeese') and not egg:return 'SHEEP' if yarn else 'COW'
    if kind=='GOOSE' and cfg.get('yarngeese') and yarn:return 'SHEEP'
    return None

def _y_fib(n):
    a,b=1,1
    for _ in range(n):a,b=b,a+b
    return a

def _y_cash(obs,action,market):
    farm=obs['farms'][int(obs['player'])];prices=obs['market']['prices'];shed=obs['private']['shed']
    try:shed=projected_shed({'farmer':action.get('farmer') or ['PASS'],'hands':action.get('hands') or [],'market':[]},FarmView(obs))
    except Exception:pass
    cash=float(farm['money']);cost=0.0;hires=int(farm.get('hires_today',0) or 0)
    quads=len(farm.get('unlocked_quadrants',[]) or [])
    for o in market:
        if not o:continue
        op=o[0]
        if op=='SELL' and len(o)>=3:
            cash+=0.8*min(max(0,int(o[2])),int(shed.get(o[1],0)))*float(prices.get(o[1],0))
        elif op=='BUY_ANIMAL' and len(o)>=3:cost+=int(o[2])*_Y_COST.get(o[1],500)
        elif op=='BUY_PRODUCT' and len(o)>=3:cost+=int(o[2])*(float(prices.get(o[1],0))+10)
        elif op=='BUY_SEED' and len(o)>=3:cost+=int(o[2])*_Y_SEED.get(o[1],100)
        elif op=='BUY_LAND':cost+=(1000,2000,4000)[min(2,max(0,quads-1))]
        elif op=='HIRE':cost+=_y_fib(hires);hires+=1
    return cash-cost

def _y_controller(obs,action,state,cfg):
    step=int(obs['step']);day=step//24;seat=int(obs['player'])
    farm=obs['farms'][seat];private=obs['private'];shed=private['shed'];inventories=private.get('inventories',[])
    shops=list((obs.get('town') or {}).get('unlocked_shops',[]) or [])
    tiles=farm['tiles'];n=len(tiles);center=n//2
    # 1. confirm last step's swapped purchases (physical shed gain)
    gained={}
    for p in state['pending']:
        to=p['to']
        if to not in gained:gained[to]=max(0,int(shed.get(to,0))-p['before'])
        got=min(p['qty'],gained[to]);gained[to]-=got
        if got>0:
            key=(p['from'],to);state['credit'][key]=state['credit'].get(key,0)+got
            if _Y_STRUCT[p['from']]!=_Y_STRUCT[to]:state['coop_swap']+=got
    state['pending']=[]
    result=_y_copy.deepcopy(action)
    market=result.get('market') or []
    result['market']=market
    # 2. purchase-point substitution by unlocked shops (cash-checked)
    if cfg['days'][0]<=day<=cfg['days'][1]:
        for o in market:
            if len(o)>=3 and o[0]=='BUY_ANIMAL' and o[1] in _Y_COST and 1<=int(o[2])<=cfg.get('maxq',2):
                to=_y_target(o[1],shops,cfg)
                if to is None or to==o[1]:continue
                trial=[list(x) if isinstance(x,list) else x for x in market]
                for t in trial:
                    if isinstance(t,list) and t==o:t[1]=to
                if _y_cash(obs,result,trial)<cfg.get('margin',100):
                    _Y_REPORT['declined']+=1;continue
                state['pending'].append({'from':o[1],'to':to,'qty':int(o[2]),'before':int(shed.get(to,0))})
                _Y_REPORT['swaps']+=int(o[2]);o[1]=to
    # 3. worker command rewrites (PICKUP / PLACE / BUILD_COOP) and harvest credit
    workers=[result.get('farmer') or ['PASS'],*(result.get('hands') or [])]
    positions=[farm['farmer'],*farm['hands']]
    avail={k:int(shed.get(k,0)) for k in _Y_COST}
    seen=set();occupied=set()
    for actor,work in enumerate(workers[:len(positions)]):
        if not work or not isinstance(work,list):continue
        inv=inventories[actor] if actor<len(inventories) else {}
        x,y=positions[actor];tile=tiles[y][x];site=(x,y);op=work[0]
        if op=='PICKUP' and len(work)>=2 and work[1] in _Y_COST:
            kind=work[1];qty=max(1,int(work[2])) if len(work)>2 else 1
            if avail.get(kind,0)>=qty:avail[kind]-=qty;continue
            if not (x in (center-1,center) and y in (center-1,center)):continue
            if any(inv.get(a,0) for a in _Y_COST):continue
            for (frm,to),c in list(state['credit'].items()):
                if frm==kind and c>=qty and avail.get(to,0)>=qty:
                    work[1]=to;state['credit'][(frm,to)]=c-qty;avail[to]-=qty;_Y_REPORT['pick']+=qty;break
        elif op=='PLACE' and len(work)>=2 and work[1] in _Y_COST:
            kind=work[1]
            if inv.get(kind,0)>0:continue
            for to in ('COW','SHEEP','GOOSE'):
                if (to!=kind and inv.get(to,0)>0 and isinstance(tile,dict) and tile.get('kind')==_Y_STRUCT[to]
                        and not tile.get('animal') and site not in occupied):
                    work[1]=to;state['sites'][site]=to;occupied.add(site);_Y_REPORT['place']+=1;break
        elif op=='BUILD_COOP' and state['coop_swap']>0:
            work[0]='BUILD_PASTURE';_Y_REPORT['coop']+=1
        elif op=='HARVEST' and site in state['sites'] and site not in seen:
            if isinstance(tile,dict) and tile.get('animal')==state['sites'][site]:
                units=max(0,int(tile.get('yield_units',0) or 0))
                if units:
                    prod=_Y_PRODUCT[tile['animal']];state['sale'][prod]=state['sale'].get(prod,0)+units;_Y_REPORT['harvest']+=units
            seen.add(site)
    result['farmer'],result['hands']=workers[0],workers[1:]
    # 4. sell the extra production at the tape's own existing sale slots (never add new orders)
    if cfg.get('boost',True):
        for prod,credit in list(state['sale'].items()):
            credit=min(credit,int(shed.get(prod,0)))
            state['sale'][prod]=credit
            if credit<=0:continue
            planned=sum(max(0,int(o[2])) for o in market if len(o)>=3 and o[0]=='SELL' and o[1]==prod)
            if planned<=0:continue
            extra=min(credit,int(shed.get(prod,0))-planned)
            if extra<=0:continue
            for o in market:
                if len(o)>=3 and o[0]=='SELL' and o[1]==prod and int(o[2])>0:
                    o[2]=int(o[2])+extra;state['sale'][prod]=credit-extra;_Y_REPORT['boost']+=extra;break
    return result

def _y_agent_shopherd(observation,configuration=None):
    action=_Y_HOST(observation,configuration)
    try:
        seat=int(observation['player']);step=int(observation['step'])
        state=_Y_STATES.get(seat)
        if state is None or step<=state['last']:state=_Y_STATES[seat]=_y_new_state()
        state['last']=step
        return _y_controller(observation,action,state,_Y_CFG)
    except Exception:
        _Y_REPORT['errors']+=1
        return action


## Build the submission

Set `USE_CADENCE` at the top of the next cell, then run it. The cell joins the V44 base and the layers in order (layer 1, layer 2, layer 3 only when `USE_CADENCE = True`, layer 4, then any `EXTRA_LAYERS`) into `main.py`, packs `submission.tar.gz` with `main.py` at the archive root, and loads `main.py` the way Kaggle's loader does (exec the file, take the last callable) to print the entry point.

The live file mixes CRLF and LF line endings between blocks, and `%%writefile` does not preserve them, so the cell restores each layer's original newlines before joining. The sha256 assert proves the result is byte-for-byte the live submission.

For your own builds, `EXTRA_LAYERS = ['my_layer']` appends `build/my_layer.py` after layer 4, and `VERIFY_LIVE_SHA = False` lets a build with an edited layer through. The live sha256 is asserted only for the default layers with `VERIFY_LIVE_SHA = True`; in every case the cell prints the sha256 and writes `main.py` and `submission.tar.gz`.

| `USE_CADENCE` | Build | Live submission | sha256 of `main.py` |
|---|---|---|---|
| `True` | V44 + layers 1, 2, 3, 4 | 56280605 | `fa9e47d81de50208...` |
| `False` | V44 + layers 1, 2, 4 | 56277542 | `016b9a32248f3447...` |

In [ ]:
USE_CADENCE = True       # True:  V44 + layers 1, 2, 3, 4 (same file as live submission 56280605)
                         # False: V44 + layers 1, 2, 4    (same file as live submission 56277542)
EXTRA_LAYERS = []        # e.g. ['my_layer'] -> build/my_layer.py is appended after layer 4
VERIFY_LIVE_SHA = True   # assert the live sha256 (default layers only; skipped with EXTRA_LAYERS)

import hashlib, os, tarfile

EXPECTED_SHA256 = {
    True: 'fa9e47d81de5020847c1dbc7ca10d058b227be66deea7c019040fb12a1d99f19',   # four-layer build
    False: '016b9a32248f34477cf5f153119d6f361c39d99a50734e435ac2860f65493fe7',  # three-layer build
}

# How each layer sits in the live main.py: (newline, leading newlines, trailing newlines).
# %%writefile writes the cell text with the platform newline and may add or keep a final
# newline, so each written layer is normalized to LF, trimmed of outer blank lines and
# restored from this table.
LAYER_FORMAT = {
    'layer1_preguard': ('\n', 1, 2),
    'layer2_lockstep': ('\r\n', 1, 1),
    'layer3_cadence':  ('\n', 2, 1),
    'layer4_yarnherd': ('\r\n', 1, 1),
}


def layer_bytes(name):
    newline, lead, trail = LAYER_FORMAT[name]
    with open(os.path.join('build', name + '.py'), 'rb') as f:
        text = f.read().decode('utf-8').replace('\r', '').strip('\n')
    return ('\n' * lead + text + '\n' * trail).replace('\n', newline).encode('utf-8')


names = ['layer1_preguard', 'layer2_lockstep'] + (['layer3_cadence'] if USE_CADENCE else [])
names += ['layer4_yarnherd']
with open(os.path.join('build', 'v44_base.py'), 'rb') as f:
    source = f.read()
print(f'  v44_base          {len(source):9,d} bytes')
for name in names:
    part = layer_bytes(name)
    source += part
    print(f'  {name:17s} {len(part):9,d} bytes')
for name in EXTRA_LAYERS:
    with open(os.path.join('build', name + '.py'), 'rb') as f:
        part = b'\n' + f.read()
    source += part
    print(f'  {name:17s} {len(part):9,d} bytes (extra layer)')

sha = hashlib.sha256(source).hexdigest()
live = not EXTRA_LAYERS and sha == EXPECTED_SHA256[USE_CADENCE]
if VERIFY_LIVE_SHA and not EXTRA_LAYERS:
    assert live, (f'main.py does not match the live file (sha256 {sha}). '
                  'If you edited a layer on purpose, set VERIFY_LIVE_SHA = False.')

with open('main.py', 'wb') as f:
    f.write(source)
with tarfile.open('submission.tar.gz', 'w:gz') as tar:
    tar.add('main.py', arcname='main.py')
with tarfile.open('submission.tar.gz') as tar:
    members = tar.getnames()

print()
if USE_CADENCE:
    build = 'four-layer (V44 + layers 1, 2, 3, 4)'
else:
    build = 'three-layer (V44 + layers 1, 2, 4)'
if EXTRA_LAYERS:
    build += ' + extra layers ' + ', '.join(EXTRA_LAYERS)
print('build:', build)
print(f'{os.path.abspath("main.py")}: {len(source):,} bytes')
status = '(matches the live file)' if live else '(custom build, not the live file)'
print(f'  sha256 {sha} {status}')
tar_size = os.path.getsize('submission.tar.gz')
print(f'{os.path.abspath("submission.tar.gz")}: {tar_size:,} bytes, members {members}')

# Load main.py the way kaggle_environments does: exec the file and take the last callable.
with open('main.py', encoding='utf-8') as f:
    raw = f.read()
env = {}
exec(compile(raw, os.path.abspath('main.py'), 'exec'), env)
entry = [v for v in env.values() if callable(v)][-1]
print('entry point (last callable):', entry.__name__)

## What did not work

- **Daily hire caps.** Capping V44's HIRE orders per day lost clearly. Cap 10 went 0-6 (-9883 mean margin) and cap 11 went 2-4 (-533). The extra hand earns about 500-1000 coins per day, while its Fibonacci hire cost is at most 377, so capping V44's hiring only loses money.
- **pipe-5 settings on V44.** Porting pipe-5's `clamp_sells` / `dead_stock` settings onto V44 went 2-4.
- **Another turn-1 opening on V44.** The turn-1 opening of a top leaderboard cluster (called C9 in a public analysis notebook) changed the result by +1 coin, so no real effect.
- **Our old wheat-pump trick.** It won by +60-70k locally against clones of tt95, an earlier public agent family, then collapsed to a 1622 rating live once the meta moved. Lesson: local wins against yesterday's agents mean little. Always test against the newest public agents and fresh live tapes.

## Next steps

These are the open ideas we see, roughly in order of value:

- **The turn-1 BUY 70 / SELL 70 wheat opener.** Some newer opponents buy and sell 70 wheat on turn 1. Two of the three-layer build's three live losses (by 1 and 663 coins) came against this opener, and the four-layer build still loses the exact tape of the 663-coin game. Understanding where the opener gains, and answering it in the first day's market, is the most obvious next layer.
- **Shop-aware herds beyond the yarn store.** Layer 4 only acts when YARN_STORE is unlocked at the day 8-11 purchase points. Other shop mixes may also call for a different herd.
- **The last-day dumps.** Some same-step endgame sells between clones are still contested. Moving them likely needs changes inside V44's terminal planner, not another market wrapper.
- **Reactive tapes.** A tape opponent that switches to V44 once its recorded actions stop making sense would make the testbed closer to live play.

## Fork it

The layers are built to stack. To add your own:

```python
# Capture the current entry point BEFORE defining any function or class.
_MY_HOST = [v for v in list(globals().values()) if callable(v)][-1]

def my_layer_agent(observation, configuration=None):
    action = _MY_HOST(observation, configuration)
    try:
        return adjust(observation, action)   # your change
    except Exception:
        return action                        # fall back to the host's action
```

Put `adjust` and any other helpers between the capture and the agent, and keep your agent function the last callable in the file.

To build it in this notebook, add a cell above **Build the submission** that starts with `%%writefile build/my_layer.py`, then set `EXTRA_LAYERS = ['my_layer']` in the build cell. Extra layers are appended after layer 4 in list order, and the build skips the live sha256 check for them.

Before you trust a layer, test it in the V44 mirror from both seats, then against pipe-4 and pipe-5, then on fresh live tapes. If you beat this build, publish it and post a link in the comments.

If this notebook helped you, please upvote it.